In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras import layers,models


In [2]:
#4 cechy: novelty, feasibility, cost, risk
X = np.array([
    [0.9,0.8,0.3,0.2],
    [0.7,0.6,0.5,0.4],
    [0.4,0.5,0.8,0.7],
    [0.85,0.75,0.2,0.3],
    [0.3,0.4,0.9,0.8]
],dtype=np.float32)

In [3]:
y = np.array([1,1,0,1,0],dtype=np.float32)

In [4]:
#atraktor dla 4 cech
attractor = np.array([0.9,0.8,0.2,0.2],dtype=np.float32)

In [5]:
#liczenie odległosci od atraktora
distances = np.linalg.norm(X-attractor,axis=1).reshape(-1,1)

In [6]:
#hybyrydowe wejscie
X_hybrid = np.hstack([X,distances])
X_hybrid

array([[0.9       , 0.8       , 0.3       , 0.2       , 0.10000001],
       [0.7       , 0.6       , 0.5       , 0.4       , 0.45825756],
       [0.4       , 0.5       , 0.8       , 0.7       , 0.9746794 ],
       [0.85      , 0.75      , 0.2       , 0.3       , 0.12247448],
       [0.3       , 0.4       , 0.9       , 0.8       , 1.17047   ]],
      dtype=float32)

In [7]:
#sieć naeuronowa
model = models.Sequential([
    layers.Dense(16,activation='relu',input_shape=(5,)),
    layers.Dense(8,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [9]:
model.fit(X_hybrid,y,epochs=50,verbose=0)

In [10]:
preds = model.predict(X_hybrid)
preds

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step


array([[0.70268863],
       [0.62761545],
       [0.54798836],
       [0.6874569 ],
       [0.5253829 ]], dtype=float32)

Porównanie treningu sieci neurnowej na danych standardowych i danych hybrydowych

In [11]:
np.random.seed(42)

# 1. Generujemy dane
X = np.random.rand(500, 4).astype(np.float32)
attractor = np.array([0.9, 0.8, 0.2, 0.2], dtype=np.float32)

In [12]:
# etykieta zależna od odległości
distances = np.linalg.norm(X - attractor, axis=1)
y = (distances < 0.45).astype(np.float32)

In [13]:

# 2. Wersja zwykła i hybrydowa
X_plain = X
X_hybrid = np.hstack([X, distances.reshape(-1, 1)])


In [14]:
# 3. Ten sam podział train/test
X_train_plain, X_test_plain, y_train, y_test = train_test_split(
    X_plain, y, test_size=0.3, random_state=42
)

X_train_hybrid, X_test_hybrid, _, _ = train_test_split(
    X_hybrid, y, test_size=0.3, random_state=42
)

In [15]:
# 4. Funkcja budująca model
def build_model(input_dim):
    model = models.Sequential([
        layers.Dense(16, activation="relu", input_shape=(input_dim,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

In [16]:
# 5. Model zwykły
model_plain = build_model(4)
model_plain.fit(X_train_plain, y_train, epochs=30, verbose=0)
pred_plain = (model_plain.predict(X_test_plain) > 0.5).astype(int)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


In [17]:
# 6. Model hybrydowy
model_hybrid = build_model(5)
model_hybrid.fit(X_train_hybrid, y_train, epochs=30, verbose=0)
pred_hybrid = (model_hybrid.predict(X_test_hybrid) > 0.5).astype(int)

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [18]:
# 7. Porównanie
acc_plain = accuracy_score(y_test, pred_plain)
acc_hybrid = accuracy_score(y_test, pred_hybrid)

print("Accuracy plain:", acc_plain)
print("Accuracy hybrid:", acc_hybrid)

Accuracy plain: 0.8866666666666667
Accuracy hybrid: 0.8866666666666667


# Hybryda -> dwa źródła decyzji
score = 0.7 * NN_prediction + 0.3*resonance

In [19]:
X = np.array([
    [0.9, 0.8, 0.3, 0.2],
    [0.7, 0.6, 0.5, 0.4],
    [0.4, 0.5, 0.8, 0.7],
    [0.85, 0.75, 0.2, 0.3],
    [0.3, 0.4, 0.9, 0.8]
], dtype=np.float32)

y = np.array([1, 1, 0, 1, 0], dtype=np.float32)

attractor = np.array([0.9, 0.8, 0.2, 0.2], dtype=np.float32)

model = models.Sequential([
    layers.Dense(16, activation="relu", input_shape=(4,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=50, verbose=0)

nn_pred = model.predict(X).flatten()

distances = np.linalg.norm(X - attractor, axis=1)
resonance = 1 / (1 + distances)

final_score = 0.7 * nn_pred + 0.3 * resonance
final_class = (final_score > 0.5).astype(int)

print("NN prediction:", nn_pred)
print("Resonance:", resonance)
print("Final score:", final_score)
print("Final class:", final_class)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
NN prediction: [0.6238524  0.5783888  0.52279437 0.65350276 0.5019575 ]
Resonance: [0.9090909  0.68574995 0.5064113  0.89088887 0.46072972]
Final score: [0.7094239  0.61059713 0.5178794  0.7247186  0.48958915]
Final class: [1 1 1 1 0]


In [21]:
from tensorflow.keras import layers, Model

# -----------------------------
# 1. Dane
# -----------------------------
X = np.array([
    [0.9, 0.8, 0.3, 0.2],
    [0.7, 0.6, 0.5, 0.4],
    [0.4, 0.5, 0.8, 0.7],
    [0.85, 0.75, 0.2, 0.3],
    [0.3, 0.4, 0.9, 0.8]
], dtype=np.float32)

y = np.array([1, 1, 0, 1, 0], dtype=np.float32).reshape(-1, 1)

# -----------------------------
# 2. Funkcja budująca model
# -----------------------------
def build_attractor_model(input_dim=4, hidden_dim=8, embedding_dim=2):
    inputs = layers.Input(shape=(input_dim,), name="input_features")

    hidden = layers.Dense(hidden_dim, activation="relu", name="hidden_dense")(inputs)

    embedding = layers.Dense(
        embedding_dim,
        activation=None,
        name="embedding"
    )(hidden)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        name="classifier_output"
    )(embedding)

    model = Model(
        inputs=inputs,
        outputs=[outputs, embedding],
        name="AttractorEmbeddingModel"
    )
    return model

# -----------------------------
# 3. Utworzenie modelu
# -----------------------------
model = build_attractor_model()

# pokaż architekturę
model.summary()

# -----------------------------
# 4. Atraktor w przestrzeni embeddingu
# -----------------------------
embedding_attractor = tf.constant([[1.0, 1.0]], dtype=tf.float32)

# -----------------------------
# 5. Optymalizator
# -----------------------------
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

# -----------------------------
# 6. Pętla treningowa
# -----------------------------
for epoch in range(100):
    with tf.GradientTape() as tape:
        preds, emb = model(X, training=True)

        classification_loss = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(y, preds)
        )

        attractor_loss = tf.reduce_mean(
            tf.reduce_sum((emb - embedding_attractor) ** 2, axis=1)
        )

        total_loss = classification_loss + 0.1 * attractor_loss

    grads = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if epoch % 20 == 0:
        print(
            f"epoch={epoch}, "
            f"class_loss={classification_loss.numpy():.4f}, "
            f"attr_loss={attractor_loss.numpy():.4f}, "
            f"total={total_loss.numpy():.4f}"
        )

# -----------------------------
# 7. Predykcja końcowa
# -----------------------------
preds, emb = model(X, training=False)

print("\nPredictions:")
print(preds.numpy())

print("\nEmbeddings:")
print(emb.numpy())

Model: "AttractorEmbeddingModel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_features (InputLayer)     │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_dense (Dense)            │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 2)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier_output (Dense)       │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61 (244.00 B)

 Trainable params: 61 (244.00 B)

 Non-trainable params: 0 (0.00 B)

epoch=0, class_loss=0.6867, attr_loss=3.9512, total=1.0819
epoch=20, class_loss=0.4706, attr_loss=0.8186, total=0.5524
epoch=40, class_loss=0.2207, attr_loss=1.3514, total=0.3558
epoch=60, class_loss=0.1366, attr_loss=1.6398, total=0.3006
epoch=80, class_loss=0.1173, attr_loss=1.5615, total=0.2735

Predictions:
[[0.9705407 ]
 [0.9411414 ]
 [0.2706598 ]
 [0.9677739 ]
 [0.08552227]]

Embeddings:
[[ 1.2080622   1.3692616 ]
 [ 0.9222034   1.1420747 ]
 [ 0.40640745 -0.4346931 ]
 [ 1.2241933   1.3187709 ]
 [ 0.18719071 -0.9999082 ]]
